In [ ]:
import os
import json
import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import precision_score, f1_score
from PIL import Image

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch Version: 2.9.0+cu126
CUDA Available: True


In [ ]:
class BasicConv2d(nn.Module):
    """Conv2D + BatchNorm + ReLU block"""
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(BasicConv2d, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class SpatialSoftmax(nn.Module):
    """Normalize attention map so sum of pixels = 1"""
    def forward(self, x):
        n, c, h, w = x.size()
        x = x.view(n, c, -1)
        x = F.softmax(x, dim=2)
        x = x.view(n, c, h, w)
        return x

In [ ]:
class FaceEncodingStream(nn.Module):
    def __init__(self):
        super(FaceEncodingStream, self).__init__()
        # 5 Convs, 4 Pools (Paper Sec 3.2.1)
        self.features = nn.Sequential(
            BasicConv2d(3, 32, 3, 1, 1), nn.MaxPool2d(2, 2),
            BasicConv2d(32, 64, 3, 1, 1), nn.MaxPool2d(2, 2),
            BasicConv2d(64, 128, 3, 1, 1), nn.MaxPool2d(2, 2),
            BasicConv2d(128, 256, 3, 1, 1), nn.MaxPool2d(2, 2),
            BasicConv2d(256, 256, 3, 1, 1)
        )
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        return x.view(x.size(0), -1)

In [ ]:
class ContextEncodingStream(nn.Module):
    def __init__(self):
        super(ContextEncodingStream, self).__init__()
        # Backbone (Same as Face Stream)
        self.features = nn.Sequential(
            BasicConv2d(3, 32, 3, 1, 1), nn.MaxPool2d(2, 2),
            BasicConv2d(32, 64, 3, 1, 1), nn.MaxPool2d(2, 2),
            BasicConv2d(64, 128, 3, 1, 1), nn.MaxPool2d(2, 2),
            BasicConv2d(128, 256, 3, 1, 1), nn.MaxPool2d(2, 2),
            BasicConv2d(256, 256, 3, 1, 1)
        )
        # Attention Module
        self.attn_conv = nn.Sequential(
            BasicConv2d(256, 128, 3, 1, 1),
            nn.Conv2d(128, 1, 3, 1, 1)
        )
        self.spatial_softmax = SpatialSoftmax()
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x_c = self.features(x) # [B, 256, H, W]

        # Calculate Attention
        attn = self.attn_conv(x_c)
        attn_map = self.spatial_softmax(attn) # [B, 1, H, W]

        # Apply Attention
        x_attended = x_c * attn_map
        x_out = self.gap(x_attended)
        return x_out.view(x_out.size(0), -1), attn_map

In [ ]:
class CAERNet(nn.Module):
    def __init__(self, num_classes=7):
        super(CAERNet, self).__init__()
        self.face_stream = FaceEncodingStream()
        self.context_stream = ContextEncodingStream()

        # Adaptive Fusion Network
        self.fc_face_atten = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1))
        self.fc_context_atten = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1))

        # self.fusion_dropout = nn.Dropout(p=0.5)

        self.classifier = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, num_classes)
        )

    def forward(self, face_img, context_img):
        x_f = self.face_stream(face_img)
        x_c, attn_map = self.context_stream(context_img)

        # Compute Fusion Weights (Lambda)
        score_f = self.fc_face_atten(x_f)
        score_c = self.fc_context_atten(x_c)
        weights = F.softmax(torch.cat([score_f, score_c], dim=1), dim=1)

        lambda_f = weights[:, 0].unsqueeze(1)
        lambda_c = weights[:, 1].unsqueeze(1)

        # Weighted Concatenation
        x_fused = torch.cat([x_f * lambda_f, x_c * lambda_c], dim=1)

        # x_fused = self.fusion_dropout(x_fused)

        logits = self.classifier(x_fused)
        return {"logits": logits, "weights": weights, "attn": attn_map}

# Test model creation
model = CAERNet(num_classes=7)
print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

Model Parameters: 2,387,402


In [ ]:
class CAERDataset(Dataset):
    def __init__(self, root_dir, phase='train', face_size=96, context_size=224):
        self.samples = []
        self.face_size = face_size
        self.context_size = context_size

        # 1. Walk through folders: root_dir/emotion_name/image.jpg
        if not os.path.exists(root_dir):
            print(f"Warning: Directory {root_dir} not found. Dataset will be empty.")
        else:
            self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
            self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

            for cls in self.classes:
                cls_path = os.path.join(root_dir, cls)
                for img_name in os.listdir(cls_path):
                    if img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
                        self.samples.append((os.path.join(cls_path, img_name), self.class_to_idx[cls]))

        # 2. Setup Transformations
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        # 3. Load Face Detector
        self.face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        # Load and convert to RGB
        img_bgr = cv2.imread(img_path)
        if img_bgr is None: return self.__getitem__((idx + 1) % len(self)) # Skip corrupt
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Detect Face
        faces = self.face_cascade.detectMultiScale(img_rgb, 1.1, 4)
        h, w = img_rgb.shape[:2]

        if len(faces) > 0:
            # Largest face
            x, y, fw, fh = max(faces, key=lambda b: b[2] * b[3])
        else:
            # Fallback: Center crop
            x, y, fw, fh = w//4, h//4, w//2, h//2

        # --- Stream A: Face Crop ---
        face_crop = img_rgb[y:y+fh, x:x+fw]
        if face_crop.size == 0: face_crop = img_rgb # Safety
        face_crop = cv2.resize(face_crop, (self.face_size, self.face_size))

        # --- Stream B: Context (Masked Face) ---
        context_img = img_rgb.copy()
        cv2.rectangle(context_img, (x, y), (x+fw, y+fh), (0, 0, 0), -1) # Mask with black
        context_img = cv2.resize(context_img, (self.context_size, self.context_size))

        return (
            self.transform(Image.fromarray(face_crop)),
            self.transform(Image.fromarray(context_img)),
            label
        )

In [ ]:
# The preprocessed class used
class PreprocessedCAERDataset(Dataset):
    def __init__(self, data_dir):
        self.files = sorted([
            os.path.join(data_dir, f)
            for f in os.listdir(data_dir)
            if f.endswith(".pt")
        ])

        if len(self.files) == 0:
            raise RuntimeError(f"No .pt files found in {data_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        face, context, label = torch.load(self.files[idx])
        return face, context, label


In [ ]:
#Preprocessing
TEST_DIR = "/content/gdrive/MyDrive/CAER-S/test"   # test images folder
SAVE_DIR = "/content/gdrive/MyDrive/preprocessed_test"  # where preprocessed test tensors will go
FACE_SIZE = 96
CONTEXT_SIZE = 224

os.makedirs(SAVE_DIR, exist_ok=True)

# Create list of all test images
all_images = []
classes = sorted([d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))])
class_to_idx = {cls: i for i, cls in enumerate(classes)}
for cls in classes:
    cls_path = os.path.join(TEST_DIR, cls)
    for img_name in os.listdir(cls_path):
        if img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
            all_images.append((os.path.join(cls_path, img_name), class_to_idx[cls]))

# Face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ---------- PREPROCESS TEST DATA ----------
for idx, (img_path, label_idx) in enumerate(all_images):
    # Load image
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f"Image not found, skipping: {img_path}")
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # Detect face
    faces = face_cascade.detectMultiScale(img_rgb, 1.1, 4)
    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda b: b[2]*b[3])
    else:
        x, y, fw, fh = w//4, h//4, w//2, h//2

    # --- Face Crop ---
    face_crop = img_rgb[y:y+fh, x:x+fw]
    if face_crop.size == 0: face_crop = img_rgb
    face_crop = cv2.resize(face_crop, (FACE_SIZE, FACE_SIZE))

    # --- Context (Masked Face) ---
    context_img = img_rgb.copy()
    cv2.rectangle(context_img, (x, y), (x+fw, y+fh), (0, 0, 0), -1)
    context_img = cv2.resize(context_img, (CONTEXT_SIZE, CONTEXT_SIZE))

    # Convert to tensor
    face_t = transform(Image.fromarray(face_crop))
    context_t = transform(Image.fromarray(context_img))

    # Save
    save_path = os.path.join(SAVE_DIR, f"{idx:05d}.pt")
    try:
        torch.save((face_t, context_t, torch.tensor(label_idx)), save_path)
    except Exception as e:
        print(f"Failed to save {img_path}: {e}")

print("Data preprocessing completed.")

In [ ]:
# Load the data into Colab's local SSD
!rsync -a --progress /content/drive/MyDrive/preprocessed /content/preprocessed_train
!rsync -a --progress /content/drive/MyDrive/preprocessed_test  /content/preprocessed_test

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Sample viewing, for verification purposes
# Paths
train_dir = "/content/preprocessed"
test_dir = "/content/preprocessed_test"

def show_samples(dataset_dir, num_samples=5):
    classes = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
    print("Classes:", classes)

    plt.figure(figsize=(15, 3))

    for i in range(num_samples):
        cls = random.choice(classes)
        cls_path = os.path.join(dataset_dir, cls)
        img_name = random.choice(os.listdir(cls_path))
        img_path = os.path.join(cls_path, img_name)

        # Read image (BGR -> RGB)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.subplot(1, num_samples, i+1)
        plt.imshow(img)
        plt.title(cls)
        plt.axis("off")

    plt.show()

# Show samples from train and test
print("Train samples:")
show_samples(train_dir)

print("Test samples:")
show_samples(test_dir)


In [ ]:
# To check the train folder's size - should be 9.2 GB
!du -sh /content/CAER-S/train

In [ ]:
def train_model(train_dir, test_dir, epochs=10, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"--- Starting Training on {device} ---")

    # ---------------------
    # Initialize Datasets
    # ---------------------
    train_dataset = PreprocessedCAERDataset(train_dir)
    test_dataset  = PreprocessedCAERDataset(test_dir)

    if len(train_dataset) == 0:
        print("No training data found.")
        return

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print("Train samples:", len(train_dataset))
    print("Test samples:", len(test_dataset))

    # ---------------------
    # Initialize Model
    # ---------------------
    num_classes = 7   # <-- IMPORTANT: set manually (CAER-S has 7 emotions)
    model = CAERNet(num_classes=num_classes).to(device)

    optimizer = optim.SGD(
        model.parameters(),
        lr=0.01,
        momentum=0.9,
        weight_decay=5e-4
    )

    criterion = nn.CrossEntropyLoss()

    metrics = {
        "train_loss": [], "train_accuracy": [], "train_precision": [],
        "val_loss": [], "val_accuracy": [], "val_precision": []
    }

    # ---------------------
    # Training Loop
    # ---------------------
    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch+1}/{epochs} ---")
        model.train()

        total_loss = 0
        correct = 0
        total = 0
        train_preds = []
        train_labels = []

        for i, (face, context, label) in enumerate(train_loader):
            face = face.to(device, non_blocking=True)
            context = context.to(device, non_blocking=True)
            label = label.to(device, non_blocking=True)

            optimizer.zero_grad()
            output = model(face, context)
            loss = criterion(output["logits"], label)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, pred = torch.max(output["logits"], 1)

            correct += (pred == label).sum().item()
            total += label.size(0)

            train_preds.extend(pred.cpu().numpy())
            train_labels.extend(label.cpu().numpy())

            if i % 50 == 0:
                print(f"Batch {i} | Loss: {loss.item():.4f}")

        avg_train_loss = total_loss / len(train_loader)
        train_acc = correct / total
        train_precision = precision_score(
            train_labels, train_preds, average="weighted", zero_division=0
        )

        metrics["train_loss"].append(avg_train_loss)
        metrics["train_accuracy"].append(train_acc)
        metrics["train_precision"].append(train_precision)

        print(
            f"TRAIN -> Loss: {avg_train_loss:.4f} | "
            f"Acc: {train_acc*100:.2f}% | "
            f"Prec: {train_precision:.4f}"
        )

        # ---------------------
        # Validation
        # ---------------------
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            for face, context, label in test_loader:
                face = face.to(device, non_blocking=True)
                context = context.to(device, non_blocking=True)
                label = label.to(device, non_blocking=True)

                output = model(face, context)
                loss = criterion(output["logits"], label)

                val_loss += loss.item()
                _, pred = torch.max(output["logits"], 1)

                val_correct += (pred == label).sum().item()
                val_total += label.size(0)

                val_preds.extend(pred.cpu().numpy())
                val_labels.extend(label.cpu().numpy())

        avg_val_loss = val_loss / len(test_loader)
        val_acc = val_correct / val_total
        val_precision = precision_score(
            val_labels, val_preds, average="weighted", zero_division=0
        )

        metrics["val_loss"].append(avg_val_loss)
        metrics["val_accuracy"].append(val_acc)
        metrics["val_precision"].append(val_precision)

        print(
            f"VAL   -> Loss: {avg_val_loss:.4f} | "
            f"Acc: {val_acc*100:.2f}% | "
            f"Prec: {val_precision:.4f}"
        )

        # Save metrics every epoch (Colab-safe)
        with open("/content/drive/MyDrive/caer_metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)

    torch.save(model.state_dict(), "/content/drive/MyDrive/caer_net_model.pth")
    print("Model saved to caer_net_model.pth")
    print("Metrics saved to caer_metrics.json")



In [ ]:
def predict_image(image_path, model_path, class_names):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load Model
    model = CAERNet(num_classes=len(class_names)).to(device)
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print("Model loaded.")
    else:
        print("Model file not found, using random weights.")
    model.eval()

    # Process Image
    preprocessor = CAERDataset(root_dir="", phase='test') # Hack to use __getitem__ logic

    # Manually run processing steps from Dataset class
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print("Image not found.")
        return
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Face Detection & Masking Logic (Same as Dataset)
    faces = preprocessor.face_cascade.detectMultiScale(img_rgb, 1.1, 4)
    h, w = img_rgb.shape[:2]
    if len(faces) > 0: x, y, fw, fh = max(faces, key=lambda b: b[2] * b[3])
    else: x, y, fw, fh = w//4, h//4, w//2, h//2

    face_crop = cv2.resize(img_rgb[y:y+fh, x:x+fw], (96, 96))
    context_img = img_rgb.copy()
    cv2.rectangle(context_img, (x, y), (x+fw, y+fh), (0, 0, 0), -1)
    context_img = cv2.resize(context_img, (224, 224))

    # To Tensor
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.485]*3, [0.229]*3)])
    face_t = transform(Image.fromarray(face_crop)).unsqueeze(0).to(device)
    context_t = transform(Image.fromarray(context_img)).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        out = model(face_t, context_t)
        probs = F.softmax(out['logits'], dim=1)
        top_prob, top_idx = torch.max(probs, 1)

    print(f"\nPrediction: {class_names[top_idx.item()]} ({top_prob.item()*100:.2f}%)")
    print(f"Fusion Weights -> Face: {out['weights'][0][0]:.2f}, Context: {out['weights'][0][1]:.2f}")

In [ ]:
def predict_pt_file(pt_path, model_path, class_names):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load model
    model = CAERNet(num_classes=len(class_names)).to(device)
    if not os.path.exists(model_path):
        print("Model file not found. Using random weights.")
    else:
        model.load_state_dict(torch.load(model_path, map_location=device))
        print("Model loaded.")
    model.eval()

    # Load preprocessed sample (tuple: face, context, label)
    if not os.path.exists(pt_path):
        print(f"PT file not found: {pt_path}")
        return

    sample = torch.load(pt_path)
    face_t    = sample[0].unsqueeze(0).to(device)  # face
    context_t = sample[1].unsqueeze(0).to(device)  # context
    true_label_idx = sample[2].item()              # label

    # Predict
    with torch.no_grad():
        out = model(face_t, context_t)
        probs = F.softmax(out["logits"], dim=1)
        top_prob, top_idx = torch.max(probs, 1)

    predicted_emotion = class_names[top_idx.item()]
    true_emotion = class_names[true_label_idx]

    print(f"True emotion: {true_emotion} ({true_label_idx})")
    print(f"Predicted emotion: {predicted_emotion} ({top_idx.item()}) | Confidence: {top_prob.item()*100:.2f}%")
    print(f"Fusion Weights -> Face: {out['weights'][0][0]:.2f}, Context: {out['weights'][0][1]:.2f}")



In [ ]:
CLASSES = ["angry", "sad", "happy", "fear", "disgust", "surprise", "neutral"]

PT_FILE = "/content/preprocessed_test/preprocessed_test/12902.pt"
MODEL_PATH = "/content/drive/MyDrive/caer_net_model.pth"

if MODE == "predict":
    predict_pt_file(PT_FILE, MODEL_PATH, CLASSES)
else:
    print("Prediction mode not selected. Change MODE to 'predict' to run inference.")

In [ ]:
MODE = "predict"  # Change to "predict" for inference

In [ ]:
# To verify which files are in the test folder

folder = "/content/preprocessed_test/preprocessed_test"
print("Files in folder:", os.listdir(folder))
print("Does the file exist?", os.path.exists(os.path.join(folder, ".pt")))


Files in folder: ['12902.pt', '20421.pt', '16164.pt', '13722.pt', '03891.pt', '10170.pt', '15246.pt', '05251.pt', '17542.pt', '03205.pt', '11092.pt', '16791.pt', '11780.pt', '01927.pt', '14307.pt', '01095.pt', '18763.pt', '12348.pt', '10837.pt', '02333.pt', '12219.pt', '09383.pt', '06419.pt', '07949.pt', '02881.pt', '00193.pt', '18140.pt', '12301.pt', '18814.pt', '10595.pt', '10441.pt', '13839.pt', '20790.pt', '10439.pt', '03928.pt', '17689.pt', '14627.pt', '18351.pt', '05231.pt', '20780.pt', '05147.pt', '15775.pt', '10434.pt', '06295.pt', '05910.pt', '02040.pt', '01432.pt', '07171.pt', '19575.pt', '05415.pt', '17006.pt', '20575.pt', '10889.pt', '08047.pt', '09691.pt', '17720.pt', '06123.pt', '04046.pt', '17094.pt', '20623.pt', '08667.pt', '02505.pt', '16740.pt', '20475.pt', '04525.pt', '10221.pt', '11350.pt', '07419.pt', '17693.pt', '20962.pt', '10806.pt', '08036.pt', '01060.pt', '04822.pt', '06679.pt', '07928.pt', '18344.pt', '03291.pt', '08418.pt', '00411.pt', '00244.pt', '14205.pt'

In [ ]:
if MODE == "train":
    train_model(train_dir="/content/preprocessed_train/preprocessed",test_dir="/content/preprocessed_test/preprocessed_test",
                epochs=30,batch_size=32
)
else:
    print("Training mode not selected. Change MODE to 'train' to begin training.")

In [ ]:
sample = torch.load("/content/preprocessed_test/preprocessed_test/13722.pt")
face, context, label = sample

plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(face.permute(1,2,0))  # face tensor
plt.title("Face Crop")

plt.subplot(1,2,2)
plt.imshow(context.permute(1,2,0))  # context tensor
plt.title("Context Image")
plt.show()


In [ ]:
# Load metrics
metrics_path = "/content/drive/MyDrive/caer_metrics.json"
with open(metrics_path, "r") as f:
    metrics = json.load(f)

# Example: plot train/val loss
plt.figure(figsize=(10, 5))
plt.plot(metrics["train_loss"], label="Train Loss")
plt.plot(metrics["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.legend()
plt.show()

# Example: plot train/val accuracy
plt.figure(figsize=(10, 5))
plt.plot(metrics["train_accuracy"], label="Train Accuracy")
plt.plot(metrics["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy over Epochs")
plt.legend()
plt.show()
